# L14 · 실제 소형 모델 학습과 서버 확장

## Goal

- download guard를 읽는다
- LoRA memory estimate를 해석한다
- laptop과 server 경계를 나눈다

## Setup

이 cell은 CPU·seed·offline 상태와 split hash를 먼저 고정합니다. toy 연산은 결정론적인 CPU 연산만 쓰며, package trainer의 전역 결정론 기본값은 유지합니다.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L14:toy:42").hexdigest()
print(f"lesson=L14 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L14 language=ko profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.10.12 rl_study=0.1.0.dev0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:266a5466f8136e9801321a6a19b4caba3ccc1bb62cf9958ffbdc632d74e61de4 data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. 현재 위치와 핵심 식

⏱ 5분 · 1/3 section · [필수/CORE]

현재 위치: toy 알고리즘 → **실제 모델 profile** → server recipe

$$M_{train}\approx M_{weights}+M_{gradients}+M_{optimizer}+M_{activations}+M_{headroom}$$

toy와 실제 공개 모델은 같은 알고리즘 API를 쓰지만 다운로드, tokenizer revision, dtype, adapter, device가 새 실패 경계가 됩니다. laptop preset은 LoRA와 보수적 headroom을 사용하고, server preset은 분산 framework 책임을 adapter 밖으로 분리합니다.

### 2. 작은 숫자로 실행

⏱ 6분 · 2/3 section · [필수/CORE]

**먼저 예측:** 모델이 cache에 없고 승인 flag도 없을 때 optional import와 download 중 무엇보다 먼저 멈춰야 하나요? 20초 동안 답을 적은 뒤 실행하세요.

<details><summary>정답 보기</summary>download guard가 optional framework import보다 먼저 멈춰야 환경 변경과 대용량 전송이 일어나지 않습니다.</details>

In [2]:
from rl_study.adapters import MODEL_PRESETS, enforce_download_guard, estimate_training_memory
from rl_study.errors import DownloadApprovalRequired
manifest = MODEL_PRESETS["laptop-smoke"]
memory = estimate_training_memory(
    manifest, adapter="lora", dtype="float32", batch_size=1, sequence_length=128
)
guard_blocked = False
try:
    enforce_download_guard(manifest, cached=False, accept_download=False)
except DownloadApprovalRequired:
    guard_blocked = True
print({"model": manifest.hub_id, "revision": manifest.revision[:12],
       "expected_download_mb": round(manifest.expected_bytes / 1e6, 1),
       "recommended_memory_gib": round(memory.recommended_bytes / 2**30, 2),
       "guard_blocked_before_download": guard_blocked})

{'model': 'HuggingFaceTB/SmolLM2-135M-Instruct', 'revision': '12fd25f77366', 'expected_download_mb': 269.1, 'recommended_memory_gib': 1.59, 'guard_blocked_before_download': True}


### 3. 구현 해부

⏱ 6분 · 3/3 section · [심화/DEEP DIVE]

**왜 이렇게 구현했나:** 정확한 peak memory는 hardware와 kernel에 따라 달라지므로 estimate와 measured 값을 구분합니다. QLoRA는 더 작은 memory 대안이지만 quantization backend 호환성이라는 새 경계를 만듭니다.

**흔한 함정:** `model_id`만 pin하고 revision을 비우면 같은 config가 다른 weights를 받을 수 있습니다. model·tokenizer revision과 예상 byte를 manifest에 함께 둡니다. 회귀 test: `test_large_download_guard_runs_before_optional_import`.

**쉬어가기:** 지금 출력한 한 값만 설명할 수 있으면 다음 cell로 가세요.

## Checks

In [3]:
assert guard_blocked and manifest.expected_bytes > 100_000_000
print("checks=passed")

checks=passed


**회상 문제:** memory estimate가 통과해도 실제 train 전 preflight가 다시 확인해야 할 세 조건은 무엇인가요? 1~2문장으로 답하세요.

## 내가 자주 틀리는 것

- loss가 유한하면 구현도 맞다고 생각한다.
- `terminated`와 `truncated`, prompt와 action을 합친다.
- 한 seed의 작은 결과를 알고리즘 순위로 확대한다.

## 60초 요약

- **실행 결론:** laptop smoke preset은 약 269.1MB download와 1.59GiB 권장 memory를 보고했고, 승인 없는 download를 실제 전송 전에 차단했습니다.
- 실제 확인: `test_large_download_guard_runs_before_optional_import`.
- 출력은 고정 seed의 toy 실행이며 논문 규모 결과가 아닙니다.

## Next Steps

1. L15에서 single response를 넘어 tool call과 여러 observation을 가진 trajectory를 학습합니다.
2. `[필수/CORE]` assertion을 한 번 깨뜨리고 오류를 읽습니다.
3. package test를 열어 notebook의 작은 식과 production guard를 연결합니다.

## Sources

- `framework-trl` — `docs/sources.yml`
- `framework-verl` — `docs/sources.yml`
- `framework-openrlhf` — `docs/sources.yml`